In [9]:
import os, json, cv2, numpy as np, matplotlib.pyplot as plt
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import masks_to_boxes

import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import functional as F

import albumentations as A # Library for augmentations
import transforms, utils, engine, train
from utils import collate_fn
from engine import train_one_epoch, evaluate
import cv2

from model_utilities import parse_annotation, precheck_annotation

import torch
import random
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from PIL import ImageDraw, Image


In [10]:
def get_model(num_keypoints, weights_path=None):
    
    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   pretrained_backbone=True,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes = 2, # Background is the first class, object is the second class
                                                                   rpn_anchor_generator=anchor_generator)

    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)        
        
    return model

In [ ]:
# test the mdoel with images that were not part of the test or train set
WEIGHTS_FILE = "./keypointsrcnn_weights_20epochs.pth"
IMG_PATH = "/Users/oliverbaltzer/Google Drive/Shared drives/Cooley_Lab/Hybrid Speckling_Spot-Vein-Project/LeahSamuels_PetalPhotos/P121/F1P121_Vein_Center_210714.jpg"


img_original = cv2.imread(IMG_PATH)
# img_original = cv2.rotate(img_original, cv2.ROTATE_90_COUNTERCLOCKWISE)
img = [F.to_tensor(img_original)]

model = get_model(4,weights_path=WEIGHTS_FILE)
model.eval()
cpu_device = torch.device("cpu")
output = model(img)
print(output)

NameError: name 'ClassDataset' is not defined

In [ ]:
def visualize_kp(image, bbox, keypoints):
    """
    visualize a specific image and keypoints
    """
    if isinstance(image, torch.Tensor):
        image = F.to_pil_image(image)
    elif isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    draw = ImageDraw.Draw(image)


    for obj_kpts in keypoints:
        for kp in obj_kpts:
            x, y = float(kp[0]), float(kp[1])
            draw.ellipse(
                [(x - 4, y - 4),
                    (x + 4, y + 4)],
                outline="red", width=2)
    # Visualize bounding box

    for box in bbox:
        x_min, y_min, x_max, y_max = box
        draw.rectangle(
            [int(x_min),int(y_min),int(x_max),int(y_max)],
            outline="blue", width=2
        )


    # Show result
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted keypoints")
    plt.show()


In [ ]:
scores = output[0]['scores'].detach().cpu().numpy()

high_scores_idxs = np.where(scores > 0.4)[0].tolist() # Indexes of boxes with scores > 0.7
if not high_scores_idxs:
    print("no viable keypoints")

post_nms_idxs = torchvision.ops.nms(output[0]['boxes'][high_scores_idxs], output[0]['scores'][high_scores_idxs], 0.3).cpu().numpy() # Indexes of boxes left after applying NMS (iou_threshold=0.3)

# Below, in output[0]['keypoints'][high_scores_idxs][post_nms_idxs] and output[0]['boxes'][high_scores_idxs][post_nms_idxs]
# Firstly, we choose only those objects, which have score above predefined threshold. This is done with choosing elements with [high_scores_idxs] indexes
# Secondly, we choose only those objects, which are left after NMS is applied. This is done with choosing elements with [post_nms_idxs] indexes

keypoints = []
for kps in output[0]['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    keypoints.append([list(map(int, kp[:2])) for kp in kps])


bboxes = []
for bbox in output[0]['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    bboxes.append(list(map(int, bbox.tolist())))
    
visualize_kp(img[0], bboxes, keypoints)

TypeError: DistanceKeypointEvaluator.update() missing 1 required positional argument: 'targets'

In [12]:
class ClassDataset(Dataset):
    def __init__(self, data, transform=None, demo=False):                
        self.transform = transform
        self.demo = demo # Use demo=True if you need transformed and original images (for example, for visualization purposes)
        self.impairs = data
    
    def __getitem__(self, idx):
        impair = self.impairs[idx]
        img_path = impair[0]
        annotations_path = impair[1]
        # print(annotations_path)
        img_original = cv2.imread(img_path)
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)
        keypoints_original = parse_annotation(annotations_path)
        bboxes_original = np.array(masks_to_boxes(torch.as_tensor([get_vein_shape(cv2.imread(PATH, 0))])))
        bboxes_labels_original = ["leaf" for _ in bboxes_original]


        # print(f"keypoints: {keypoints_original}")
        # print(f"original kp shape: {np.shape(keypoints_original)}")
        # print(f"bounding boxes: {bboxes_original}")
        # print(f"bounding box labels: {bboxes_labels_original}")


        if self.transform:   
            # Converting keypoints from [x,y,visibility]-format to [x, y]-format + Flattening nested list of keypoints            
            # For example, if we have the following list of keypoints for three objects (each object has two keypoints):
            # [[obj1_kp1, obj1_kp2], [obj2_kp1, obj2_kp2], [obj3_kp1, obj3_kp2]], where each keypoint is in [x, y]-format            
            # Then we need to convert it to the following list:
            # [obj1_kp1, obj1_kp2, obj2_kp1, obj2_kp2, obj3_kp1, obj3_kp2]
            # print(f"original keypoints{keypoints_original}")
            keypoints_original_flattened = [el[0:2] for kp in keypoints_original for el in kp]
            # print(f"original flattened keypoints{keypoints_original_flattened}")
            
            # Apply augmentations
            transformed = self.transform(image=img_original, bboxes=bboxes_original, bboxes_labels=bboxes_labels_original, keypoints=keypoints_original_flattened)
            img = transformed['image']
            bboxes = transformed['bboxes']
            
            # Unflattening list transformed['keypoints']
            # For example, if we have the following list of keypoints for three objects (each object has two keypoints):
            # [obj1_kp1, obj1_kp2, obj2_kp1, obj2_kp2, obj3_kp1, obj3_kp2], where each keypoint is in [x, y]-format
            # Then we need to convert it to the following list:
            # [[obj1_kp1, obj1_kp2], [obj2_kp1, obj2_kp2], [obj3_kp1, obj3_kp2]]
            keypoints_transformed_unflattened = np.reshape(np.array(transformed['keypoints']), (1,4,2)).tolist()
            # print(f"transformed unflattened: {keypoints_transformed_unflattened}")
            # Converting transformed keypoints from [x, y]-format to [x,y,visibility]-format by appending original visibilities to transformed coordinates of keypoints
            keypoints = []
            # print(f"original keypoints: {keypoints_original}")
            for o_idx, obj in enumerate(keypoints_transformed_unflattened): # Iterating over objects
                obj_keypoints = []
                # print(obj)
                # print(keypoints_original[o_idx])
                for k_idx, kp in enumerate(obj): # Iterating over keypoints in each object
                    # kp - coordinates of keypoint
                    # keypoints_original[o_idx][k_idx][2] - original visibility of keypoint
                    obj_keypoints.append(kp + [keypoints_original[o_idx][k_idx][2]])
                keypoints.append(obj_keypoints)
        
        else:
            img, bboxes, keypoints = img_original, bboxes_original, keypoints_original        
        
        # Convert everything into a torch tensor        
        bboxes = torch.as_tensor(bboxes, dtype=torch.float32)       
        target = {}
        target["boxes"] = bboxes
        target["labels"] = torch.as_tensor([1 for _ in bboxes], dtype=torch.int64)
        target["image_id"] = int(idx)
        target["area"] = (bboxes[:, 3] - bboxes[:, 1]) * (bboxes[:, 2] - bboxes[:, 0])
        target["iscrowd"] = torch.zeros(len(bboxes), dtype=torch.int64)
        target["keypoints"] = torch.as_tensor(keypoints, dtype=torch.float32)        
        img = F.to_tensor(img)
        
        bboxes_original = torch.as_tensor(bboxes_original, dtype=torch.float32)
        target_original = {}
        target_original["boxes"] = bboxes_original
        target_original["labels"] = torch.as_tensor([1 for _ in bboxes_original], dtype=torch.int64)
        target_original["image_id"] = int(idx)
        target_original["area"] = (bboxes_original[:, 3] - bboxes_original[:, 1]) * (bboxes_original[:, 2] - bboxes_original[:, 0])
        target_original["iscrowd"] = torch.zeros(len(bboxes_original), dtype=torch.int64)
        target_original["keypoints"] = torch.as_tensor(keypoints_original, dtype=torch.float32)        
        img_original = F.to_tensor(img_original)

        if self.demo:
            return img, target, img_original, target_original
        else:
            return img, target
    
    def __len__(self):
        return len(self.impairs)

In [ ]:
KEYPOINTS_FOLDER_TRAIN = '/Users/oliverbaltzer/Google Drive/Shared drives/Cooley_Lab/Hybrid Speckling_Spot-Vein-Project/JoshuaShin_PetalPhotos/'
dataset = ClassDataset(grab_all_data(KEYPOINTS_FOLDER_TRAIN), transform=train_transform(), demo=True)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)


iterator = iter(data_loader_test)
images, targets = next(iterator)
images = list(image.to(device) for image in images)

with torch.no_grad():
    model.to(device)
    model.eval()
    output = model(images)

print("Predictions: \n", output)


NameError: name 'grab_all_data' is not defined